# NBM v5 MaxT/MinT Verification
Point verification of NBM QMD probabilistic temperature forecasts for a NWS CWA.
Covers deterministic skill, percentile reliability, and threshold-exceedance reliability.

In [ ]:
import sys
import math
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import dask
import dask.array as da
import requests
from pathlib import Path
from datetime import date
from scipy.spatial import cKDTree
from eccodes import (
    CodesInternalError, codes_grib_new_from_file, codes_get_values, codes_release,
)

sys.path.insert(0, r'C:\Users\Michael.wessler\Code\nbm-v5-verification')
from analysis.read_nbm_forecasts import open_nbm_maxt_mint, NBM_PARA_ROOT
import nbm_grib_tools as nt

In [ ]:
# ── Study period ──────────────────────────────────────────
START_DATE = date(2026, 3, 1)
END_DATE   = date(2026, 5, 1)

# ── CWA & API credentials ─────────────────────────────
CWA            = "PHI"
SYNOPTIC_TOKEN = "a2386b75ecbc4c2784db1270695dde73"

# ── Paths ────────────────────────────────────────────
OBS_DIR = Path(r"N:\data\nbm_para\observations")

# ── Forecast mean method ───────────────────────────────
# 'qmd'    — trapezoidal integration over the quantile density function
# 'simple' — unweighted mean of the 8 available percentile values
MEAN_METHOD = 'simple'

# ── Valid-time offsets (hours past obs file base date → NBM valid_time) ───
# MaxT: obs file day D → valid_time D+1 06Z  (+30 h)
# MinT: obs file day D → valid_time D 18Z    (+18 h)
VAR_HOURS = {'maxt': 30, 'mint': 18}

# ── Station matching ─────────────────────────────────────────
TOL_DEG    = 0.05               # max station↔obs tolerance (~5 km)
KEEP_MNETS = {"1", "2", "153"} # ASOS/AWOS (1), RAWS (2), GHCN-Daily (153)

# ── Per-station distribution histogram: lead day to display ───────────
LEAD_DAY = 1

# ── Probabilistic verification ─────────────────────────────────
SKIP_PERCS  = {100}                    # p=100 excluded from reliability diagram
PROB_BINS   = np.linspace(0, 1, 11)   # reliability diagram bin edges
BIN_CENTERS = 0.5 * (PROB_BINS[:-1] + PROB_BINS[1:])

# ── Time-matching diagnostic (Section 5) ─────────────────────────
DIAG_STATION   = 'KPHL'
DIAG_START     = '2026-03-15'  # event-day window start (inclusive)
DIAG_END       = '2026-03-28'  # event-day window end   (inclusive)
DIAG_LEAD      = 1             # lead_day to show
DIAG_INIT_HOUR = 12            # restrict to one init cycle (0, 12, or None = all)

## 1. Load Forecast Dataset
Open the NBM QMD GRIB2 archive and compute estimated deterministic means.

In [ ]:
ds = open_nbm_maxt_mint(start_date=START_DATE, end_date=END_DATE)

_probs = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 1.00])

for var in ['maxt', 'mint']:
    # QMD mean: trapezoidal rule  E[X] = trapz over [0.05,1.00] + rect for [0,0.05]
    ds[f'{var}_qmd_mean'] = (
        xr.apply_ufunc(
            np.trapz, ds[var],
            input_core_dims=[['percentile']],
            kwargs={'x': _probs},
            dask='parallelized', output_dtypes=[np.float32],
        )
        + 0.05 * ds[var].sel(percentile=5)
    )
    ds[f'{var}_simple_mean'] = ds[var].mean(dim='percentile')
    ds[f'{var}_mean'] = ds[f'{var}_{MEAN_METHOD}_mean']
    ds[f'{var}_mean'].attrs = {
        **ds[var].attrs,
        'long_name': f'{var.upper()} 2-m temperature ({MEAN_METHOD} mean)',
    }

print(f"Dataset: {dict(ds.dims)}  |  mean method: '{MEAN_METHOD}'")
ds

## 2. Station Metadata & Grid Matching
Fetch active stations from the Synoptic API, match against obs-file stations, apply network filter, then map to the nearest NBM grid pixel.

In [ ]:
resp = requests.get(
    "https://api.synopticdata.com/v2/stations/metadata",
    params={
        "token":  SYNOPTIC_TOKEN,
        "cwa":    CWA,
        "status": "active",
        "output": "json",
    },
    timeout=30,
)
resp.raise_for_status()

stations = pd.DataFrame([
    {
        "stid":    s["STID"],
        "name":    s["NAME"],
        "lat":     float(s["LATITUDE"]),
        "lon":     float(s["LONGITUDE"]),
        "network": s.get("MNET_SHORTNAME", s.get("MNET_ID", "")),
        "elev_m":  s.get("ELEVATION"),
    }
    for s in resp.json()["STATION"]
]).sort_values("stid").reset_index(drop=True)

print(f"{CWA}: {len(stations)} active stations")
stations

# Resolve network names (MNET_ID → shortname)
nets_resp = requests.get(
    "https://api.synopticdata.com/v2/networks",
    params={"token": SYNOPTIC_TOKEN, "output": "json"},
    timeout=30,
)
nets_resp.raise_for_status()
nets = {str(n["ID"]): n["SHORTNAME"] for n in nets_resp.json()["MNET"]}
stations["network_name"] = stations["network"].astype(str).map(nets).fillna("unknown")

# Load unique obs stations from one file (schema is consistent across all files)
obs_stns = (
    pd.read_csv(next(OBS_DIR.glob("obs_maxtmint_*.csv")), usecols=["sid", "name", "lat", "lon"], encoding="latin-1")
    .drop_duplicates("sid")
    .reset_index(drop=True)
)

# KD-tree nearest-neighbour: match each broad PHI Synoptic station to obs
obs_tree = cKDTree(obs_stns[["lat", "lon"]].values)
dists, idxs = obs_tree.query(stations[["lat", "lon"]].values, k=1)

matched = dists <= TOL_DEG

stations_obs = stations[matched].copy().reset_index(drop=True)
stations_obs["obs_sid"]  = obs_stns.loc[idxs[matched], "sid"].values
stations_obs["dist_deg"] = dists[matched].round(4)

# Network filter: ASOS/AWOS (1), RAWS (2), GHCN-Daily (153)
stations_filtered = (
    stations_obs[stations_obs["network"].astype(str).isin(KEEP_MNETS)]
    .reset_index(drop=True)
)

stations_filtered

print(f"{CWA}: {len(stations)} active | {len(stations_obs)} matched obs | "
      f"{len(stations_filtered)} after network filter")
stations_filtered

In [ ]:
# ── 1. Find nearest grid point (yi, xi) for every station ──────────────────
lat2d = ds['maxt_mean'].coords['lat'].values   # (nj, ni)
lon2d = ds['maxt_mean'].coords['lon'].values
nj, ni = lat2d.shape

grid_tree = cKDTree(np.column_stack([lat2d.ravel(), lon2d.ravel()]))

stn_coords = np.column_stack([
    stations_filtered['lat'].values,
    stations_filtered['lon'].values % 360,   # grid is 0-360
])
_, flat_idx = grid_tree.query(stn_coords, k=1)
yi_arr, xi_arr = np.unravel_index(flat_idx, (nj, ni))

stations_filtered = stations_filtered.copy()
stations_filtered['grid_y'] = yi_arr
stations_filtered['grid_x'] = xi_arr

print(f"Grid: {nj} × {ni} pixels  |  {len(stations_filtered)} stations mapped")

## 3. Load Observations
Read obs CSV files for the study period, melt to long form, and assign valid_times matching the NBM forecast grid.

In [ ]:
keep_sids = set(stations_filtered['obs_sid'].dropna().astype(str))

# Map obs_sid → stid so obs_df shares both identifiers with fcst_df
sid_to_stid = (
    stations_filtered[['obs_sid', 'stid']]
    .dropna(subset=['obs_sid'])
    .astype({'obs_sid': str})
    .set_index('obs_sid')['stid']
    .to_dict()
)

chunks = []
for csv_path in sorted(OBS_DIR.glob("obs_maxtmint_*.csv")):
    date_str = csv_path.stem.split('_')[-1]   # YYYYMMDD = calendar day of the extreme
    base_date = pd.Timestamp(date_str)

    df = pd.read_csv(
        csv_path,
        usecols=['sid', 'maxt', 'mint'],
        encoding='latin-1',
        dtype={'sid': str},
    )
    df = df[df['sid'].isin(keep_sids)].copy()
    if df.empty:
        continue

    # Coerce to numeric — obs files use 'M' or other flags for missing
    df['maxt'] = pd.to_numeric(df['maxt'], errors='coerce')
    df['mint'] = pd.to_numeric(df['mint'], errors='coerce')

    # Melt to long: one row per (station, variable) with correct valid_time
    df = df.melt(id_vars='sid', value_vars=['maxt', 'mint'],
                 var_name='variable', value_name='obs_value')
    df['valid_time'] = df['variable'].map(
        {v: base_date + pd.Timedelta(hours=h) for v, h in VAR_HOURS.items()}
    )
    chunks.append(df)

obs_df = (
    pd.concat(chunks, ignore_index=True)
    .rename(columns={'sid': 'obs_sid'})
    .assign(stid=lambda d: d['obs_sid'].map(sid_to_stid))
    [['stid', 'obs_sid', 'variable', 'valid_time', 'obs_value']]
    .sort_values(['stid', 'variable', 'valid_time'])
    .reset_index(drop=True)
)

print(f"{len(obs_df):,} obs rows  |  {obs_df['stid'].nunique()} stations  |  "
      f"{obs_df['valid_time'].nunique()} unique valid_times")
print(f"Observations: {len(obs_df):,} rows | {obs_df['stid'].nunique()} stations | "
      f"{obs_df['valid_time'].min().date()} → {obs_df['valid_time'].max().date()}")

## 4. Extract Forecast Points & Pair with Observations
Materialise mean point values with dask, convert K → °F, then inner-join with obs on station + valid_time.

In [ ]:
# ── 2. Lazy vectorised point extraction ────────────────────────────────────
# isel with DataArrays of indices adds a new 'station' dim; no data read yet
stn_coord = xr.DataArray(stations_filtered['stid'].values, dims='station')
y_idx = xr.DataArray(yi_arr, dims='station')
x_idx = xr.DataArray(xi_arr, dims='station')

maxt_pts = ds['maxt_mean'].isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
mint_pts = ds['mint_mean'].isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
# shape: (init_time, f_hour, station) — still lazy

# ── 3. Single dask pass for both variables ─────────────────────────────────
print("Extracting point values …")
maxt_vals, mint_vals = dask.compute(maxt_pts, mint_pts)
print("Done.")

# ── 4. Flatten to a tidy DataFrame ─────────────────────────────────────────
def _to_df(da, col):
    df = da.to_dataframe(name=col).reset_index()
    return df[['init_time', 'f_hour', 'valid_time', 'station', col]]

fcst_df = (
    _to_df(maxt_vals, 'maxt_mean_k')
    .merge(_to_df(mint_vals, 'mint_mean_k'),
           on=['init_time', 'f_hour', 'valid_time', 'station'])
)

# K → °F
fcst_df['maxt_mean_f'] = (fcst_df['maxt_mean_k'] - 273.15) * 1.8 + 32.0
fcst_df['mint_mean_f'] = (fcst_df['mint_mean_k'] - 273.15) * 1.8 + 32.0
fcst_df = fcst_df.drop(columns=['maxt_mean_k', 'mint_mean_k'])

# Attach obs_sid so we can join against obs data later
fcst_df = fcst_df.merge(
    stations_filtered[['stid', 'obs_sid', 'network_name']],
    left_on='station', right_on='stid',
).drop(columns='stid')

# ── Pair with observations ────────────────────────────────────
def _make_paired(fcst_df, obs_df, fcst_col, obs_var):
    """
    Inner-join one variable's forecast values against matched observations.

    fcst_col : column name in fcst_df  ('maxt_mean_f' | 'mint_mean_f')
    obs_var  : value in obs_df.variable ('maxt' | 'mint')

    Returns a DataFrame with columns:
        station, init_time, f_hour, valid_time, lead_day,
        fcst_f, obs_f
    """
    fcst = (
        fcst_df[['station', 'init_time', 'f_hour', 'valid_time', fcst_col]]
        .copy()
        .rename(columns={fcst_col: 'fcst_f'})
    )
    fcst['fcst_f']     = pd.to_numeric(fcst['fcst_f'],     errors='coerce')
    fcst['valid_time'] = pd.to_datetime(fcst['valid_time'])
    fcst['init_time']  = pd.to_datetime(fcst['init_time'])
    fcst = fcst.dropna(subset=['fcst_f'])   # drop rows where this variable has no data

    obs = (
        obs_df[obs_df['variable'] == obs_var][['stid', 'valid_time', 'obs_value']]
        .rename(columns={'stid': 'station', 'obs_value': 'obs_f'})
        .copy()
    )
    obs['obs_f']       = pd.to_numeric(obs['obs_f'],       errors='coerce')
    obs['valid_time']  = pd.to_datetime(obs['valid_time'])
    obs = obs.dropna(subset=['obs_f'])

    paired = fcst.merge(obs, on=['station', 'valid_time'], how='inner')
    paired['lead_day'] = (
        paired['valid_time'].dt.normalize()
        - paired['init_time'].dt.normalize()
    ).dt.days

    return paired.sort_values(['station', 'init_time', 'f_hour']).reset_index(drop=True)


maxt_paired = _make_paired(fcst_df, obs_df, 'maxt_mean_f', 'maxt')
mint_paired = _make_paired(fcst_df, obs_df, 'mint_mean_f', 'mint')

for nm, df in [('maxt', maxt_paired), ('mint', mint_paired)]:
    err = df['fcst_f'] - df['obs_f']
    print(f"{nm}_paired: {len(df):,} rows | {df['station'].nunique()} stns | "
          f"bias={err.mean():+.2f}°F  MAE={err.abs().mean():.2f}°F")


## 5. Time-Matching Verification Diagnostic
Verify obs and forecast are correctly aligned in time for a single station.
Event day = calendar day the extreme occurred (= obs CSV file date = valid_time − VAR_HOURS offset).

**Pairing convention** (`VAR_HOURS = {'maxt': 30, 'mint': 18}`):
- MaxT for event day D: obs file → `obs_maxtmint_D.csv`, valid_time → D+1 06Z (+30 h)
- MinT for event day D: obs file → `obs_maxtmint_D.csv`, valid_time → D 18Z (+18 h)

Set `DIAG_INIT_HOUR=12` for a clean single-cycle series.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC C — Time-matching verification
#
# X-axis: *event day* = the calendar day the temperature extreme occurred
#         = obs CSV file date = valid_time − VAR_HOURS offset.
# DIAG_START / DIAG_END are event-day bounds (not valid_time bounds).
#
# Pairing convention (VAR_HOURS = {'maxt': 30, 'mint': 18}):
#   MaxT for event day D:
#     obs_file    → obs_maxtmint_D.csv
#     valid_time  → D+1 06Z  (base_date + 30 h)
#     fcst_period → D 12Z → D+1 06Z  (f018 from 12Z D, lead_day=1)
#
#   MinT for event day D:
#     obs_file    → obs_maxtmint_D.csv
#     valid_time  → D 18Z   (base_date + 18 h)
#     fcst_period → (D−1) 12Z → D 18Z  (f030 from 12Z D−1, lead_day=1)
#
# DIAG_INIT_HOUR: 0, 12, or None.  For lead_day=1 both the 12Z init
#   (f018 for MaxT) and the 00Z init (f030 for MaxT) map to the same
#   event day and valid_time → with None both appear → zigzag pattern.
#   Set DIAG_INIT_HOUR=12 for a clean single-cycle series.
# ══════════════════════════════════════════════════════════════════════════════


def _window(df, station, start, end, lead, var_offset_h, init_hour=None):
    """
    Filter paired DataFrame for one station/lead/init cycle.

    Filtering uses *event_day* = valid_time − var_offset_h (the calendar day
    the temperature extreme occurred = obs CSV file date), so DIAG_START /
    DIAG_END refer to the event day, not the NBM valid_time.

    Adds 'event_day' column (midnight timestamps) to the returned DataFrame.
    """
    df2 = df.copy()
    df2['event_day'] = (
        pd.to_datetime(df2['valid_time']) - pd.Timedelta(hours=var_offset_h)
    ).dt.normalize()

    mask = (
        (df2['station']    == station) &
        (df2['lead_day']   == lead) &
        (df2['event_day']  >= pd.Timestamp(start)) &
        (df2['event_day']  <= pd.Timestamp(end))
    )
    if init_hour is not None:
        mask &= (pd.to_datetime(df2['init_time']).dt.hour == init_hour)
    return df2[mask].sort_values('event_day').reset_index(drop=True)

mx = _window(maxt_paired, DIAG_STATION, DIAG_START, DIAG_END, DIAG_LEAD,
             VAR_HOURS['maxt'], DIAG_INIT_HOUR)
mn = _window(mint_paired, DIAG_STATION, DIAG_START, DIAG_END, DIAG_LEAD,
             VAR_HOURS['mint'], DIAG_INIT_HOUR)

init_label = f"{DIAG_INIT_HOUR:02d}Z init only" if DIAG_INIT_HOUR is not None else "all init cycles"
print(f"Station : {DIAG_STATION}   Lead day : {DIAG_LEAD}   ({init_label})")

# ── Duplicate event_day check ──────────────────────────────────────────────
for name, df in [('MaxT', mx), ('MinT', mn)]:
    n_ed   = df['event_day'].nunique()
    n_rows = len(df)
    utc_hrs = sorted(df['valid_time'].dt.hour.unique())
    f_hrs   = sorted(df['f_hour'].unique())
    dup_msg = (f"  ⚠  {n_rows - n_ed} duplicate event_days — "
               "try setting DIAG_INIT_HOUR=12 or =0 to isolate one cycle"
               if n_rows > n_ed else "  ✓ no duplicates")
    print(f"  {name}: {n_rows} rows | {n_ed} unique event days | "
          f"valid_time UTC hrs {utc_hrs} | f_hours {f_hrs}{dup_msg}")

# ── Plot (x = event day, i.e. the calendar day the temperature occurred) ───
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)

for ax, df, var_lbl, utc_hr, color_fcst in [
    (axes[0], mx, 'MaxT', 6,  'tomato'),
    (axes[1], mn, 'MinT', 18, 'steelblue'),
]:
    ax.plot(df['event_day'], df['obs_f'],
            'o-', color='black', ms=5, lw=1.4, alpha=0.85, label='Obs',  zorder=3)
    ax.plot(df['event_day'], df['fcst_f'],
            's--', color=color_fcst, ms=5, lw=1.4, alpha=0.9, label='NBM fcst', zorder=3)

    ax.set_title(
        f'{var_lbl}  —  {DIAG_STATION},  lead day {DIAG_LEAD}  '
        f'(valid @ {utc_hr:02d}Z,  {init_label})',
        fontsize=10, fontweight='bold')
    ax.set_ylabel('°F', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(ls='--', alpha=0.35)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # Annotate valid_time and init cycle for each marker
    for _, row in df.drop_duplicates('event_day').iterrows():
        init_hr = pd.to_datetime(row['init_time']).hour
        vt      = pd.to_datetime(row['valid_time'])
        ax.annotate(
            f"{vt.strftime('%HZ')}\n(init {init_hr:02d}Z f{int(row['f_hour'])})",
            xy=(row['event_day'], row['obs_f']),
            xytext=(0, 8), textcoords='offset points',
            fontsize=4.5, color='gray', ha='center', linespacing=1.3)

fig.suptitle(
    f'Time-matching diagnostic — {DIAG_STATION}   {DIAG_START} → {DIAG_END}  '
    f'(x = event day)',
    fontsize=11, fontweight='bold')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# Matched table
# 'date' = event day (calendar day the temperature extreme occurred = obs file date).
# Both MaxT and MinT for the same event day share the same 'date' value and
# merge into one row, even though their valid_times differ.
# ══════════════════════════════════════════════════════════════════════════════
def _tbl_half(df, prefix, obs_hour_offset):
    """
    obs_hour_offset : hours added to the obs file's base_date to assign
                      valid_time  (30 for maxt, 18 for mint).
    date (merge key): event day = valid_time − obs_hour_offset, normalized.
    """
    out = df[['valid_time', 'init_time', 'f_hour', 'fcst_f', 'obs_f']].copy()
    vt = pd.to_datetime(out['valid_time'])
    it = pd.to_datetime(out['init_time'])

    out[f'{prefix}_fcst_valid']  = vt.dt.strftime('%Y-%m-%d %HZ')
    out[f'{prefix}_obs_valid']   = vt.dt.strftime('%Y-%m-%d %HZ')   # same — join key
    # obs CSV file that provided this value (file date = event day):
    out[f'{prefix}_obs_file']    = (vt - pd.Timedelta(hours=obs_hour_offset)).dt.strftime('obs_maxtmint_%Y%m%d.csv')
    out[f'{prefix}_fcst_period'] = (it.dt.strftime('%m-%d %HZ') + ' → ' + vt.dt.strftime('%m-%d %HZ'))
    out[f'{prefix}_f_hour']      = out['f_hour'].astype(int)
    out[f'{prefix}_obs_f']       = out['obs_f'].round(1)
    out[f'{prefix}_fcst_f']      = out['fcst_f'].round(1)
    out[f'{prefix}_err']         = (out['fcst_f'] - out['obs_f']).round(1)
    # date = event day (same as obs file date, not valid_time date)
    out['date'] = (vt - pd.Timedelta(hours=obs_hour_offset)).dt.normalize()
    return out.drop(columns=['valid_time', 'init_time', 'f_hour', 'fcst_f', 'obs_f'])

mx_tbl = _tbl_half(mx, 'maxt', obs_hour_offset=VAR_HOURS['maxt'])
mn_tbl = _tbl_half(mn, 'mint', obs_hour_offset=VAR_HOURS['mint'])

tbl = mx_tbl.merge(mn_tbl, on='date', how='outer').sort_values('date')
tbl['date'] = tbl['date'].dt.strftime('%Y-%m-%d')

col_order = [
    'date',
    # MaxT columns
    'maxt_fcst_period', 'maxt_fcst_valid', 'maxt_obs_valid', 'maxt_obs_file',
    'maxt_f_hour', 'maxt_obs_f', 'maxt_fcst_f', 'maxt_err',
    # MinT columns
    'mint_fcst_period', 'mint_fcst_valid', 'mint_obs_valid', 'mint_obs_file',
    'mint_f_hour', 'mint_obs_f', 'mint_fcst_f', 'mint_err',
]
print(f"\nMatched table — {DIAG_STATION}, lead_day={DIAG_LEAD}, {init_label}:")
print(tbl[col_order].to_string(index=False))


## 6. Deterministic Skill
MAE, RMSE, and bias aggregated across all paired station-days.

In [ ]:
subset = maxt_paired[maxt_paired['lead_day'] == LEAD_DAY]

stations_order = sorted(subset['station'].dropna().unique())
n = len(stations_order)

ncols = 5
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8),
                         constrained_layout=True)
axes_flat = axes.flatten()

all_obs  = subset['obs_f'].dropna()
all_fcst = subset['fcst_f'].dropna()
global_min = float(min(all_obs.min(), all_fcst.min()))
global_max = float(max(all_obs.max(), all_fcst.max()))
bins = np.linspace(global_min, global_max, 20)

for i, stid in enumerate(stations_order):
    ax = axes_flat[i]
    s = subset[subset['station'] == stid]

    obs_vals  = s['obs_f'].dropna()
    fcst_vals = s['fcst_f'].dropna()

    hist_kw = dict(bins=bins, density=True, edgecolor='white', linewidth=0.4)
    ax.hist(obs_vals,  **hist_kw, color='steelblue', alpha=0.75, label='Obs')
    ax.hist(fcst_vals, **hist_kw, color='tomato',    alpha=0.65, label='NBM')

    bias = (fcst_vals.values - obs_vals.values).mean() if len(obs_vals) == len(fcst_vals) else float('nan')
    ax.set_title(stid, fontsize=9, fontweight='bold')
    ax.set_xlabel('MaxT (°F)', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.text(0.97, 0.95,
            f'n={len(obs_vals)}\nbias={bias:+.1f}°F',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=6.5, color='gray', linespacing=1.4)

axes_flat[0].legend(fontsize=7, framealpha=0.7)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(
    f'{CWA} CWA — MaxT Distribution: Obs vs NBM ({MEAN_METHOD} mean)  |  Lead Day {LEAD_DAY}  (Mar–May 2026)',
    fontsize=11, fontweight='bold')
plt.show()


In [ ]:
def skill_by_fhour(paired):
    """Compute MAE, RMSE, bias aggregated across all stations per f_hour."""
    err = paired['fcst_f'] - paired['obs_f']
    return (
        paired.assign(err=err)
        .groupby('f_hour')
        .agg(
            mae  =('err', lambda e: np.abs(e).mean()),
            rmse =('err', lambda e: np.sqrt((e**2).mean())),
            bias =('err', 'mean'),
            n    =('err', 'count'),
        )
        .reset_index()
    )

stats_maxt = skill_by_fhour(maxt_paired)
stats_mint = skill_by_fhour(mint_paired)

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True,
                         sharey=False)

METRICS = [
    ('mae',  'MAE',  'steelblue',  '-o'),
    ('rmse', 'RMSE', 'tomato',     '-s'),
    ('bias', 'Bias', 'seagreen',   '-^'),
]

for ax, stats, title in zip(axes,
                             [stats_maxt, stats_mint],
                             ['MaxT', 'MinT']):
    for col, label, color, fmt in METRICS:
        ax.plot(stats['f_hour'], stats[col],
                fmt, color=color, ms=5, lw=1.6,
                label=label)

    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(f'{title}  —  {CWA} CWA  (all stations, Mar–May 2026)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Forecast Hour', fontsize=9)
    ax.set_ylabel('°F', fontsize=9)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(24))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(12))
    ax.tick_params(labelsize=8)
    ax.grid(axis='both', which='major', ls='--', alpha=0.3)
    ax.legend(fontsize=8)

    # Annotate sample size at each point along the MAE line
    for _, row in stats.iterrows():
        ax.annotate(f"n={int(row['n'])}",
                    xy=(row['f_hour'], row['mae']),
                    xytext=(0, 6), textcoords='offset points',
                    ha='center', fontsize=5.5, color='steelblue')

plt.show()


In [ ]:
# Lead-time bands (by forecast hour)

def skill_by_date(paired, fh_min=None, fh_max=None):
    """
    Condense f_hour dimension: for each valid_time date, average errors
    across all stations within the given f_hour band.

    fh_min / fh_max : inclusive lower / exclusive upper f_hour bound.
                      Pass None to skip that side of the filter.
    """
    df = paired.copy()
    if fh_min is not None:
        df = df[df['f_hour'] > fh_min]
    if fh_max is not None:
        df = df[df['f_hour'] <= fh_max]
    df['err'] = df['fcst_f'] - df['obs_f']
    df['date'] = df['valid_time'].dt.normalize()
    return (
        df.groupby('date')
        .agg(
            mae  =('err', lambda e: np.abs(e).mean()),
            rmse =('err', lambda e: np.sqrt((e**2).mean())),
            bias =('err', 'mean'),
            n    =('err', 'count'),
        )
        .reset_index()
    )

# 2 × 2 grid:  rows = MaxT / MinT,  cols = short / medium leads
panels = [
    # (paired_df,   fh_min, fh_max, row_label,  col_label)
    (maxt_paired, None,  72,   'MaxT', 'Short leads (f≤72 h)'),
    (maxt_paired, 72,   None,  'MaxT', 'Medium leads (f>72 h)'),
    (mint_paired, None,  72,   'MinT', 'Short leads (f≤72 h)'),
    (mint_paired, 72,   None,  'MinT', 'Medium leads (f>72 h)'),
]

METRICS = [
    ('mae',  'MAE',  'steelblue', '-o', 4),
    ('rmse', 'RMSE', 'tomato',    '-s', 4),
    ('bias', 'Bias', 'seagreen',  '-^', 4),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8), constrained_layout=True, sharex=True)

for (paired, fh_min, fh_max, var_lbl, lead_lbl), ax in zip(panels, axes.flat):
    ts = skill_by_date(paired, fh_min, fh_max)

    for col, label, color, fmt, ms in METRICS:
        ax.plot(ts['date'], ts[col], fmt,
                color=color, ms=ms, lw=1.4, alpha=0.85, label=label)

    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
    ax.set_title(f'{var_lbl}  —  {lead_lbl}', fontsize=10, fontweight='bold')
    ax.set_ylabel('°F', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(axis='both', which='major', ls='--', alpha=0.3)
    ax.legend(fontsize=8, loc='upper left')

    # Sample-count bars on right twin axis
    ax2 = ax.twinx()
    ax2.bar(ts['date'], ts['n'], width=0.8, color='gray', alpha=0.12, zorder=0)
    ax2.set_ylabel('n (pairs/day)', fontsize=7, color='gray')
    ax2.tick_params(labelsize=6, colors='gray')
    ax2.spines['right'].set_color('lightgray')

# Shared x-axis formatting on bottom row only
for ax in axes[-1]:
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.suptitle(
    f'{CWA} CWA — Skill vs Date  (Mar–May 2026, all stations)',
    fontsize=12, fontweight='bold')
plt.show()


## 7. Probabilistic Verification — Percentile Reliability
Extract QMD percentile values at station grid pixels, pair with obs, and plot reliability diagrams.
A perfectly calibrated system lies on the 1:1 diagonal.

In [ ]:
# ── Extract point values for all percentiles ───────────────────────────────
# Reuses y_idx / x_idx / stn_coord already defined in the point-extraction cell
maxt_perc_pts = ds['maxt'].isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
mint_perc_pts = ds['mint'].isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
# dims: (init_time, f_hour, percentile, station) — still lazy

print("Extracting percentile point values …")
maxt_perc_vals, mint_perc_vals = dask.compute(maxt_perc_pts, mint_perc_pts)
print("Done.")

def _perc_to_df(da, col):
    df = da.to_dataframe(name=col).reset_index()
    df[col] = (df[col] - 273.15) * 1.8 + 32.0  # K → °F
    return df[['init_time', 'f_hour', 'valid_time', 'percentile', 'station', col]]

perc_df = (
    _perc_to_df(maxt_perc_vals, 'maxt_f')
    .merge(
        _perc_to_df(mint_perc_vals, 'mint_f'),
        on=['init_time', 'f_hour', 'valid_time', 'percentile', 'station'],
    )
    .merge(
        stations_filtered[['stid', 'obs_sid']],
        left_on='station', right_on='stid',
    )
    .drop(columns='stid')
)

# ── Join with obs; flag whether obs fell below each percentile threshold ────
def _make_perc_paired(perc_df, obs_df, fcst_col, obs_var):
    obs = (
        obs_df[obs_df['variable'] == obs_var][['stid', 'valid_time', 'obs_value']]
        .rename(columns={'stid': 'station', 'obs_value': 'obs_f'})
        .copy()
    )
    obs['obs_f']      = pd.to_numeric(obs['obs_f'], errors='coerce')
    obs['valid_time'] = pd.to_datetime(obs['valid_time'])
    obs = obs.dropna(subset=['obs_f'])

    fcst = perc_df[['station', 'init_time', 'f_hour', 'valid_time', 'percentile', fcst_col]].copy()
    fcst['valid_time'] = pd.to_datetime(fcst['valid_time'])
    fcst = fcst.dropna(subset=[fcst_col])

    paired = fcst.merge(obs, on=['station', 'valid_time'], how='inner')
    paired['below'] = (paired['obs_f'] < paired[fcst_col]).astype(float)
    paired['lead_day'] = (
        pd.to_datetime(paired['valid_time']).dt.normalize()
        - pd.to_datetime(paired['init_time']).dt.normalize()
    ).dt.days
    return paired.sort_values(['station', 'init_time', 'f_hour', 'percentile']).reset_index(drop=True)

maxt_perc_paired = _make_perc_paired(perc_df, obs_df, 'maxt_f', 'maxt')
mint_perc_paired = _make_perc_paired(perc_df, obs_df, 'mint_f', 'mint')

for name, df in [('maxt_perc_paired', maxt_perc_paired), ('mint_perc_paired', mint_perc_paired)]:
    print(f"{name}: {len(df):,} rows | {df['station'].nunique()} stations | "
          f"percentiles: {sorted(df['percentile'].unique())}")


In [ ]:
def reliability(paired, fh_min=None, fh_max=None):
    """
    For each percentile, compute the fraction of cases where obs < fcst threshold.
    A perfectly calibrated system lies on the 1:1 diagonal.
    """
    df = paired[~paired['percentile'].isin(SKIP_PERCS)].copy()
    if fh_min is not None:
        df = df[df['f_hour'] > fh_min]
    if fh_max is not None:
        df = df[df['f_hour'] <= fh_max]
    return (
        df.groupby('percentile')
        .agg(obs_freq=('below', 'mean'), n=('below', 'count'))
        .reset_index()
        .assign(nominal=lambda d: d['percentile'] / 100)
    )

panels = [
    (maxt_perc_paired, None, 72,  'MaxT', 'Short leads (f≤72 h)'),
    (maxt_perc_paired, 72,  None, 'MaxT', 'Medium leads (f>72 h)'),
    (mint_perc_paired, None, 72,  'MinT', 'Short leads (f≤72 h)'),
    (mint_perc_paired, 72,  None, 'MinT', 'Medium leads (f>72 h)'),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)

for (paired_data, fh_min, fh_max, var_lbl, lead_lbl), ax in zip(panels, axes.flat):
    rel = reliability(paired_data, fh_min, fh_max)

    # Reference line
    ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.45, label='Perfect calibration')

    # Reliability curve
    ax.plot(rel['nominal'], rel['obs_freq'],
            'o-', color='steelblue', ms=7, lw=1.8, zorder=3, label='NBM QMD')

    # Shading: overdispersed (above diagonal) vs underdispersed (below)
    ax.fill_between([0, 1], [0, 1], 1, color='#d9eaf7', alpha=0.25, zorder=0)
    ax.fill_between([0, 1], 0, [0, 1], color='#fde8d8', alpha=0.25, zorder=0)
    ax.text(0.82, 0.08, 'Under-\nforecast', ha='center', fontsize=7, color='#c47a3b', alpha=0.7)
    ax.text(0.18, 0.92, 'Over-\nforecast', ha='center', fontsize=7, color='#3b7ac4', alpha=0.7)

    # Annotate n per point
    for _, row in rel.iterrows():
        n_lbl = f"{int(row['n']) // 1000}k" if row['n'] >= 1000 else str(int(row['n']))
        ax.annotate(n_lbl, xy=(row['nominal'], row['obs_freq']),
                    xytext=(5, 3), textcoords='offset points',
                    fontsize=6, color='steelblue')

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Nominal probability (percentile / 100)', fontsize=9)
    ax.set_ylabel('Observed frequency (fraction below)', fontsize=9)
    ax.set_title(f'{var_lbl}  —  {lead_lbl}', fontsize=10, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(ls='--', alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle(
    f'{CWA} CWA — Reliability Diagram  (Mar–May 2026, all stations)',
    fontsize=12, fontweight='bold')
plt.show()


## 8. Probabilistic Verification — Threshold Exceedance
Extract P(T > threshold) probability messages, pair with obs, and plot reliability diagrams.
X-axis: NBM issued probability. Y-axis: observed exceedance rate. Perfect calibration = 1:1 diagonal.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Threshold-exceedance probability extraction
# Reads P(T > threshold_k) from the probability messages in maxt_qmd / mint_qmd.
# Probabilities are stored as 0–100 integers in GRIB2; normalised to [0, 1] on read.
# ══════════════════════════════════════════════════════════════════════════════
    CodesInternalError, codes_grib_new_from_file, codes_get_values, codes_release,
)

# ── 1. Discover probability message structure from sample files ──────────────
def _discover_prob_map(sample_file: Path) -> dict:
    """Return {threshold_k_float: 1-based_msg_id} for Probability messages."""
    df = nt.index_nbm5_grib(str(sample_file), convert_imperial=False)
    rows = df[df['param_type'] == 'Probability'].dropna(subset=['threshold'])
    return {float(row.threshold): int(row.msg_id) for row in rows.itertuples()}

_first_it = pd.Timestamp(ds.coords['init_time'].values[0])
for _fh in ds.coords['f_hour'].values:
    _sample_maxt = (NBM_PARA_ROOT / _first_it.strftime('%Y%m%d')
                    / f'{_first_it.hour:02d}' / f'maxt_qmd_f{int(_fh):03d}.grib2')
    if _sample_maxt.exists():
        break
for _fh in ds.coords['f_hour'].values:
    _sample_mint = (NBM_PARA_ROOT / _first_it.strftime('%Y%m%d')
                    / f'{_first_it.hour:02d}' / f'mint_qmd_f{int(_fh):03d}.grib2')
    if _sample_mint.exists():
        break

prob_map_maxt = _discover_prob_map(_sample_maxt)
prob_map_mint = _discover_prob_map(_sample_mint)

def _k_to_f(k): return (k - 273.15) * 1.8 + 32.0

print("MaxT probability thresholds:")
for k in sorted(prob_map_maxt): print(f"  {k:.4f} K = {_k_to_f(k):.0f} °F  [msg {prob_map_maxt[k]}]")
print("MinT probability thresholds:")
for k in sorted(prob_map_mint): print(f"  {k:.4f} K = {_k_to_f(k):.0f} °F  [msg {prob_map_mint[k]}]")

# ── 2. Dask-delayed reader for probability planes ────────────────────────────
@dask.delayed
def _read_prob_planes(filepath, prob_to_msg, sorted_thresholds, grid_shape):
    """Read probability messages from one GRIB2 file.
    Returns float32 (n_thr, nj, ni) with values in [0, 1] (divided by 100)."""
    from eccodes import (CodesInternalError, codes_grib_new_from_file,
                         codes_get_values, codes_release)
    n_thr = len(sorted_thresholds)
    nj_, ni_ = grid_shape
    result = np.full((n_thr, nj_ * ni_), np.nan, dtype=np.float32)
    msg_to_idx = {prob_to_msg[t]: i for i, t in enumerate(sorted_thresholds)}
    max_msg = max(prob_to_msg[t] for t in sorted_thresholds)
    with open(filepath, 'rb') as fh:
        count = 0
        while count < max_msg:
            gid = codes_grib_new_from_file(fh)
            if gid is None:
                break
            count += 1
            if count in msg_to_idx:
                idx = msg_to_idx[count]
                try:
                    vals = codes_get_values(gid).astype(np.float32)
                    vals[vals > 1.0e10] = np.nan   # eccodes fill sentinel
                    result[idx] = vals / 100.0     # 0–100 → 0–1
                except CodesInternalError:
                    pass
            codes_release(gid)
    return result.reshape(n_thr, nj_, ni_)

# ── 3. Build dask-backed DataArrays by reconstructing file paths from ds ────
def _build_prob_da(var_name, prob_map):
    product_prefix  = {'maxt': 'maxt_qmd', 'mint': 'mint_qmd'}[var_name]
    sorted_thresholds = tuple(sorted(prob_map))
    n_thr   = len(sorted_thresholds)
    grid    = (nj, ni)
    init_arr  = ds.coords['init_time'].values
    fhour_arr = ds.coords['f_hour'].values

    slabs_init = []
    for it in init_arr:
        it_dt    = pd.Timestamp(it)
        slabs_fh = []
        for fh in fhour_arr:
            fp = (NBM_PARA_ROOT / it_dt.strftime('%Y%m%d')
                  / f'{it_dt.hour:02d}' / f'{product_prefix}_f{int(fh):03d}.grib2')
            if fp.exists():
                delayed = _read_prob_planes(str(fp), prob_map, sorted_thresholds, grid)
                arr = da.from_delayed(delayed, shape=(n_thr, nj, ni), dtype=np.float32)
            else:
                arr = da.full((n_thr, nj, ni), np.nan, dtype=np.float32)
            slabs_fh.append(arr[np.newaxis, np.newaxis])   # (1, 1, n_thr, nj, ni)
        slabs_init.append(da.concatenate(slabs_fh, axis=1))   # (1, n_fhour, n_thr, nj, ni)

    full = da.concatenate(slabs_init, axis=0)   # (n_init, n_fhour, n_thr, nj, ni)
    valid_times = (
        init_arr[:, np.newaxis].astype('datetime64[h]')
        + fhour_arr[np.newaxis, :].astype('timedelta64[h]')
    ).astype('datetime64[ns]')

    return xr.DataArray(
        full,
        dims=['init_time', 'f_hour', 'threshold', 'y', 'x'],
        coords={
            'init_time':  ('init_time',             init_arr),
            'f_hour':     ('f_hour',                fhour_arr.astype(np.int32)),
            'valid_time': (['init_time', 'f_hour'], valid_times),
            'threshold':  ('threshold',             np.array(sorted_thresholds, dtype=np.float64)),
            'lat':        (['y', 'x'],              lat2d),
            'lon':        (['y', 'x'],              lon2d),
        },
        name=f'{var_name}_prob',
        attrs={'long_name': f'P({var_name.upper()} > threshold)', 'units': 'fraction (0-1)'},
    )

maxt_prob_pts = _build_prob_da('maxt', prob_map_maxt).isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
mint_prob_pts = _build_prob_da('mint', prob_map_mint).isel(y=y_idx, x=x_idx).assign_coords(station=stn_coord)
# dims: (init_time, f_hour, threshold, station) — still lazy

print("Extracting probability point values …")
maxt_prob_vals, mint_prob_vals = dask.compute(maxt_prob_pts, mint_prob_pts)
print("Done.")

# ── 4. Flatten to tidy DataFrames ────────────────────────────────────────────
def _prob_to_df(da_vals, prob_col):
    df = da_vals.to_dataframe(name=prob_col).reset_index()
    return df[['init_time', 'f_hour', 'valid_time', 'threshold', 'station', prob_col]]

maxt_prob_df = (
    _prob_to_df(maxt_prob_vals, 'prob')
    .merge(stations_filtered[['stid', 'obs_sid']], left_on='station', right_on='stid')
    .drop(columns='stid')
    .dropna(subset=['prob'])
)
mint_prob_df = (
    _prob_to_df(mint_prob_vals, 'prob')
    .merge(stations_filtered[['stid', 'obs_sid']], left_on='station', right_on='stid')
    .drop(columns='stid')
    .dropna(subset=['prob'])
)

for _name, _df in [('maxt_prob_df', maxt_prob_df), ('mint_prob_df', mint_prob_df)]:
    thresholds_f = [f'{_k_to_f(k):.0f}°F' for k in sorted(_df['threshold'].unique())]
    print(f"{_name}: {len(_df):,} rows | {_df['station'].nunique()} stations | "
          f"thresholds: {thresholds_f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Threshold-exceedance reliability diagrams
# X-axis : NBM issued probability  P(T > threshold)   [0, 1]
# Y-axis : observed exceedance rate (fraction of paired cases where obs > threshold)
# Perfect calibration lies on the 1:1 diagonal.
# ══════════════════════════════════════════════════════════════════════════════


def _make_prob_paired(prob_df, obs_df, obs_var, fh_min=None, fh_max=None):
    """Join probability forecasts with obs; compute obs_above flag."""
    obs = (
        obs_df[obs_df['variable'] == obs_var][['stid', 'valid_time', 'obs_value']]
        .rename(columns={'stid': 'station', 'obs_value': 'obs_f'})
        .copy()
    )
    obs['obs_f']      = pd.to_numeric(obs['obs_f'], errors='coerce')
    obs['valid_time'] = pd.to_datetime(obs['valid_time'])
    obs = obs.dropna(subset=['obs_f'])

    fcst = prob_df[['station', 'init_time', 'f_hour', 'valid_time', 'threshold', 'prob']].copy()
    fcst['valid_time'] = pd.to_datetime(fcst['valid_time'])
    if fh_min is not None:
        fcst = fcst[fcst['f_hour'] > fh_min]
    if fh_max is not None:
        fcst = fcst[fcst['f_hour'] <= fh_max]

    paired = fcst.merge(obs, on=['station', 'valid_time'], how='inner')
    paired['threshold_f'] = (paired['threshold'] - 273.15) * 1.8 + 32.0
    paired['obs_above']   = (paired['obs_f'] > paired['threshold_f']).astype(float)
    return paired

def _reliability_thresh(paired_df):
    """Bin by forecast probability; compute observed exceedance rate per threshold."""
    rows = []
    for thr_k, grp in paired_df.groupby('threshold'):
        thr_f = round((thr_k - 273.15) * 1.8 + 32.0, 0)
        bin_idx = np.clip(np.digitize(grp['prob'].values, PROB_BINS) - 1,
                          0, len(BIN_CENTERS) - 1)
        for b in range(len(BIN_CENTERS)):
            mask = bin_idx == b
            n = int(mask.sum())
            if n == 0:
                continue
            rows.append({
                'threshold_f': thr_f,
                'bin_center':  BIN_CENTERS[b],
                'obs_freq':    float(grp['obs_above'].values[mask].mean()),
                'n':           n,
            })
    return pd.DataFrame(rows)

panels = [
    (maxt_prob_df, 'maxt', None, 72,   'MaxT', 'Short leads (f≤72 h)'),
    (maxt_prob_df, 'maxt', 72,   None, 'MaxT', 'Medium leads (f>72 h)'),
    (mint_prob_df, 'mint', None, 72,   'MinT', 'Short leads (f≤72 h)'),
    (mint_prob_df, 'mint', 72,   None, 'MinT', 'Medium leads (f>72 h)'),
]

COLORS = ['steelblue', 'tomato', 'seagreen', 'darkorchid', 'goldenrod']

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)

for (p_df, obs_var, fh_min, fh_max, var_lbl, lead_lbl), ax in zip(panels, axes.flat):
    paired = _make_prob_paired(p_df, obs_df, obs_var, fh_min, fh_max)
    rel    = _reliability_thresh(paired)

    ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.45, label='Perfect calibration')
    ax.fill_between([0, 1], [0, 1], 1, color='#d9eaf7', alpha=0.25, zorder=0)
    ax.fill_between([0, 1], 0, [0, 1], color='#fde8d8', alpha=0.25, zorder=0)
    ax.text(0.82, 0.08, 'Under-\nforecast', ha='center', fontsize=7, color='#c47a3b', alpha=0.7)
    ax.text(0.18, 0.92, 'Over-\nforecast', ha='center', fontsize=7, color='#3b7ac4', alpha=0.7)

    for thr, color in zip(sorted(rel['threshold_f'].unique()), COLORS):
        sub = rel[rel['threshold_f'] == thr].sort_values('bin_center')
        n_total = int(sub['n'].sum())
        ax.plot(sub['bin_center'], sub['obs_freq'],
                'o-', color=color, ms=6, lw=1.6, zorder=3,
                label=f'T > {int(thr)}°F  (n={n_total:,})')
        for _, r in sub.iterrows():
            ax.annotate(
                f"{int(r['n'])//1000}k" if r['n'] >= 1000 else str(int(r['n'])),
                xy=(r['bin_center'], r['obs_freq']),
                xytext=(4, 3), textcoords='offset points',
                fontsize=5.5, color=color, alpha=0.7,
            )

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Forecast probability  P(T > threshold)', fontsize=9)
    ax.set_ylabel('Observed exceedance rate', fontsize=9)
    ax.set_title(f'{var_lbl}  —  {lead_lbl}', fontsize=10, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(ls='--', alpha=0.3)
    ax.legend(fontsize=7.5, loc='upper left')

fig.suptitle(
    f'{CWA} CWA — Threshold-Exceedance Reliability  (Mar–May 2026, all stations)',
    fontsize=12, fontweight='bold')
plt.show()
